# TIB final523: scoring the application-summarizer outputs (ROUGE and MiniCheck)

Scores `PRIVATE_applm_outputs.jsonl` from the generation notebook. ROUGE-1/2/L F1 uses scorer v2's exact
configuration (rouge-score 0.1.2; it reproduced all 2,088 stored final523 ROUGE-L values with difference 0).
MiniCheck is pinned and run exactly as in the LED/BART faithfulness protocol. BERTScore (scorer v2, xlm-roberta-large)
needs the formal scorer venv and is a separate optional step. Failed generation rows stay NULL.

**Choose a T4 GPU runtime.** Inputs in Drive `MyDrive/TIB_FINAL523_APPLM_261031/`: the code bundle `tib-final523-applm-code-261031-v1.zip` (the same one as
for generation) and `results/PRIVATE_applm_outputs.jsonl`. Every record is checkpointed; run again to resume.

In [ ]:
import hashlib, json, platform, subprocess, sys, time, zipfile
from datetime import datetime, timezone
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR = Path("/content/drive/MyDrive/TIB_FINAL523_APPLM_261031")
    ROOT = Path("/content/applm"); ROOT.mkdir(exist_ok=True)
    with zipfile.ZipFile(DRIVE_DIR / "tib-final523-applm-code-261031-v1.zip") as bundle:
        bundle.extractall(ROOT)
    INPUT_DIR = DRIVE_DIR
    OUT_DIR = DRIVE_DIR / "results"
else:
    here = Path.cwd()
    ROOT = next(p for p in [here, *here.parents] if (p / "experiments/scripts/tib_final523_applm.py").exists())
    INPUT_DIR = ROOT / "research-data/tib-final523-scoring-261024-v1"
    OUT_DIR = ROOT / "research-data/tib-final523-applm-261031-v1"  # private: holds source and summary text
OUT_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(ROOT))

PROTOCOL_PATH = ROOT / "experiments/registry/protocols/tib-final523-applm-261031-v1/protocol.json"
EXPECTED_PROTOCOL_SHA256 = "af8c9412070b0ad83ad677eabf1e527bbf06a647003e7f91ed32664e65f54820"
actual = hashlib.sha256(PROTOCOL_PATH.read_bytes()).hexdigest()
assert actual == EXPECTED_PROTOCOL_SHA256, (
    f"Protocol hash {actual} differs from the notebook's {EXPECTED_PROTOCOL_SHA256}: the code zip in Drive is "
    "probably an older build. Upload the latest tib-final523-applm-code-261031-v1.zip, restart the session and run again. Do not edit the protocol.")
print("protocol sha256 verified:", actual[:16], "...")
OUTPUTS = OUT_DIR / "PRIVATE_applm_outputs.jsonl"
assert OUTPUTS.exists(), "Run the generation notebook in final mode first"
OUTPUTS_SHA256 = hashlib.sha256(OUTPUTS.read_bytes()).hexdigest()
print("outputs:", OUTPUTS.name, OUTPUTS_SHA256[:16], "...")

In [ ]:
import importlib.metadata as md_
def version(name):
    try:
        return md_.version(name)
    except md_.PackageNotFoundError:
        return None
ENVIRONMENT = {
    "python": platform.python_version(), "platform": platform.platform(),
    "packages": {n: version(n) for n in ["nltk", "scikit-learn", "scipy", "numpy", "pandas", "torch", "transformers"]},
    "code_sha256": {p.name: hashlib.sha256(p.read_bytes()).hexdigest()
                    for p in sorted((ROOT / "experiments/scripts").glob("tib_*.py"))},
}
print(json.dumps(ENVIRONMENT["packages"], indent=1))

In [ ]:
import torch
assert torch.cuda.is_available(), "Select a GPU runtime (T4)"
ENVIRONMENT["gpu"] = torch.cuda.get_device_name(0)
print(ENVIRONMENT["gpu"], torch.__version__)

## Install the scorers (pinned)

In [ ]:
INSTALL_LOG = {}
for label, spec in (("rouge-score", "rouge-score==0.1.2"), ("MiniCheck", "minicheck @ git+https://github.com/Liyan06/MiniCheck.git@b58b9fa69acbd1015ec970fa65dd752413a053d2")):
    result = subprocess.run([sys.executable, "-m", "pip", "install", "-q", spec], capture_output=True, text=True)
    INSTALL_LOG[label] = {"returncode": result.returncode, "tail": (result.stderr or result.stdout)[-400:]}
    assert result.returncode == 0, INSTALL_LOG[label]["tail"]
import nltk  # recent NLTK needs punkt_tab, which MiniCheck does not download
for resource in ("punkt", "punkt_tab"):
    nltk.download(resource, quiet=True)
print("installed")

## Load the generated rows

In [ ]:
from collections import Counter
from experiments.scripts import tib_final523_applm as ap
from experiments.scripts import tib_final523_faithfulness as ff
GENERATED = ap.latest_rows(OUTPUTS)  # a retried row appears twice in the file; the last line wins
assert len(GENERATED) == 523 and len({r["record_id"] for r in GENERATED}) == 523, "need one row per test ID"
SCOREABLE = [r for r in GENERATED if r["scoreable"]]
print("generated:", dict(Counter(r["generation_status"] for r in GENERATED)))

## ROUGE (CPU, exact scorer v2 configuration)

In [ ]:
ROUGE = []
for r in GENERATED:
    scores = ap.rouge_scores(r["delivered_summary"], r["reference"]) if r["scoreable"] else \
        {"rouge1_f1": None, "rouge2_f1": None, "rougeL_f1": None}
    ROUGE.append({"record_id": r["record_id"], "stratum": r["source_stratum"], **scores})
print("rouge scored:", sum(1 for x in ROUGE if x["rougeL_f1"] is not None))

## MiniCheck (checkpointed; run again to resume)
Sentence level against the whole source, contract and S/M/L views, as in the LED/BART protocol. No score is printed.

In [ ]:
score_claims, MINICHECK_INFO = ff.load_minicheck_claims()
MINICHECK_INFO["install"] = "minicheck @ git+https://github.com/Liyan06/MiniCheck.git@b58b9fa69acbd1015ec970fa65dd752413a053d2"
ROWS = [ap.to_faithfulness_row(r) for r in SCOREABLE]
CHECKPOINT = OUT_DIR / "applm_faithfulness_scores.jsonl"
DETAIL_PATH = OUT_DIR / "PRIVATE_applm_sentence_scores.jsonl"  # sentence text: keep private
SPENT_PATH = OUT_DIR / "faithfulness_seconds.json"
CEILING = 10800
state = json.loads(SPENT_PATH.read_text()) if SPENT_PATH.exists() else {"seconds": 0.0, "records": 0,
                                                                      "first_score_utc": datetime.now(timezone.utc).isoformat()}
started, spent_before, records_before = time.monotonic(), state["seconds"], state["records"]

class CeilingReached(Exception):
    pass

def progress(scored, count=[0]):
    count[0] += 1
    state["seconds"] = spent_before + time.monotonic() - started
    state["records"] = records_before + count[0]
    SPENT_PATH.write_text(json.dumps(state))
    per_record = state["seconds"] / state["records"]
    if count[0] % 25 == 0:
        print(count[0], "new records,", round(state["seconds"] / 3600, 2), "GPU h")
    if state["seconds"] + per_record > CEILING:
        raise CeilingReached(f"the next record would pass the ceiling at {state['seconds'] / 3600:.2f} h")

try:
    SCORES = ff.score_rows(ROWS, score_claims, CHECKPOINT, DETAIL_PATH, on_row=progress)
except CeilingReached as stop:
    raise SystemExit(f"{stop}. Every scored record is checkpointed. Ask for more budget, then run again.")
errors = [r for r in SCORES if "views" not in r]
assert not errors, f"{len(errors)} error rows ({errors[0]['error']}): restart the runtime and run again to retry them"

## Receipt (no text)

In [ ]:
receipt = {"schema_version": 1, "protocol_id": ap.PROTOCOL_ID, "protocol_sha256": EXPECTED_PROTOCOL_SHA256,
           "outputs_file": OUTPUTS.name, "outputs_sha256": OUTPUTS_SHA256, "environment": ENVIRONMENT,
           "install_log": INSTALL_LOG, "minicheck": MINICHECK_INFO, "first_score_utc": state["first_score_utc"],
           "receipt_written_utc": datetime.now(timezone.utc).isoformat(), "faithfulness_seconds_total": state["seconds"],
           "generation": ap.generation_receipt(GENERATED), "rouge": ROUGE,
           "faithfulness_scores": sorted(SCORES, key=lambda r: r["record_id"]),
           "sentence_detail_file": DETAIL_PATH.name}
path = OUT_DIR / "applm_scores_receipt.json"
path.write_text(json.dumps(receipt, indent=2), encoding="utf-8")
print(path, hashlib.sha256(path.read_bytes()).hexdigest())

## What to send back
`applm_scores_receipt.json` (per-record ROUGE and MiniCheck scores; no text) and the executed notebook.
`PRIVATE_applm_sentence_scores.jsonl` stays on the private Drive folder and the local machine.